In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
canopy_2015 = pd.read_csv("/content/drive/MyDrive/NIST_Preprocessed/canopy_2015_Labeled.csv")
canopy_2016 = pd.read_csv("/content/drive/MyDrive/NIST_Preprocessed/canopy_2016_Labeled.csv")

In [4]:
canopy_2017 = pd.read_csv("/content/drive/MyDrive/NIST_Preprocessed/canopy_2017_Labeled.csv")
canopy_2018 = pd.read_csv("/content/drive/MyDrive/NIST_Preprocessed/canopy_2018_Labeled.csv")

In [ ]:
canopy_2015["I_string_mean"].isna().sum()

np.int64(159)

In [5]:
# Convert the TIMESTAMP column to pandas datetime object
canopy_2015['TIMESTAMP'] = pd.to_datetime(canopy_2015['TIMESTAMP'])
canopy_2016['TIMESTAMP'] = pd.to_datetime(canopy_2016['TIMESTAMP'])

# Set TIMESTAMP as index
canopy_2015.set_index('TIMESTAMP', inplace=True)
canopy_2016.set_index('TIMESTAMP', inplace=True)

In [6]:
# Convert the TIMESTAMP column to pandas datetime object
canopy_2017['TIMESTAMP'] = pd.to_datetime(canopy_2017['TIMESTAMP'])
canopy_2018['TIMESTAMP'] = pd.to_datetime(canopy_2018['TIMESTAMP'])

# Set TIMESTAMP as index
canopy_2017.set_index('TIMESTAMP', inplace=True)
canopy_2018.set_index('TIMESTAMP', inplace=True)

#CS-LSTM Regression Description

> **Reference Paper:**  
> *"A Comprehensive Framework for Predictive Maintenance in Solar Photovoltaic Systems Using Contextual Deep Model"* (Baruah et al., 2025)

---

## Objective

The goal of the **CS-LSTM** model is to predict the expected **AC power output (`InvPAC_kW_Avg`)** under **normal operating conditions** by combining:

- Operational inverter measurements
- Contextual information (seasonality and time-of-day)

Unlike a standard LSTM, CS-LSTM incorporates contextual information through a **Kronecker product expansion**, allowing the model to learn context-dependent relationships.

The prediction residual

$$
\text{Residual} = \text{Measured Power} - \text{Predicted Power}
$$

is later used as the **anomaly signal** for **Stage 2 (XGBoost fault classifier).**

---

## Mathematical Formulation

### Standard LSTM
$$
\text{gate} = \sigma(W_h h_{t-1} + W_x x_t + b)
$$

### Contextual Sequence LSTM (CS-LSTM)

$$
\text{gate} = \sigma(W_h h_{t-1} + B(x_t \otimes \Gamma(z_t)) + b)
$$

where

- **$x_t$** → Primary operational features at timestep *t*
- **$z_t$** → Contextual features (season & time)
- **$Gamma(z_t)$** → Polynomial basis expansion of the contextual variables
- **$\otimes$** → Kronecker product

---

### Intuition

Instead of learning a single fixed relationship between the inputs and the output, CS-LSTM learns relationships that **depend on the operating context**.

For example:

- The importance of irradiance at **12:00 in July**
- is **different** from the importance of irradiance at **12:00 in December**.

The contextual expansion enables the network to adapt its behavior according to the season and time of day.

---

## Hyperparameter Search Space

```python
Info = {
    # Dataset split
    "train_years": [2015, 2016],
    "val_year": 2017,
    "test_year": 2018,

    # Operating mask
    "irr_threshold": 100.0,   # W/m²

    # Primary Features (x_t)
    "primary_features": [
        "POA_fused",          # POA irradiance
        "AmbTemp_C_Avg",      # Ambient temperature
        "module_temp",        # Module temperature
        "InvVDCin_Avg",       # DC voltage
        "InvIDCin_Avg",       # DC current
        "InvPDC_kW_Avg",      # DC power
        "InvPAC_kW_Avg_lag1"  # Previous AC power
    ],

    # Target
    "target": "InvPAC_kW_Avg",

    # Context Expansion
    "poly_degree": [2, 3],

    # Model
    "seq_len": [5, 10, 15, 30, 60],
    "hidden_size": [10, 15, 20, 25, 30],
    "num_layers": [1, 2],
    "dropout": [0.1, 0.2, 0.3],

    # Training
    "batch_size": [64, 128, 256],
    "lr": [1e-5, 1e-3],
    "epochs": [50, 150],
    "weight_decay": 1e-6,
}
```

---

## NIST Canopy Specifications

```python
P_AC_RATED_KW = 260.0   # Inverter rated AC power (kW)
P_DC_NOM_KW   = 243.0   # DC nameplate power (kW)
```

#Step 1: Fault Labels

Rather than merging `label_string`/`label_inverter` into a single priority-ordered
column (which would hide genuine co-occurrence between fault types), we keep
both label sets fully separate and add lightweight boolean indicators on top.
This supports both a coarse "fault vs no-fault" detection task and full
subsystem-level classification, without discarding information.

In [7]:
#2015
NON_FAULT_VALUES = ['NORMAL', 'NOT_EVALUABLE', 'MISSING_DATA', 'INSUFFICIENT_DATA']

canopy_2015['has_string_fault']   = ~canopy_2015['label_string'].isin(NON_FAULT_VALUES)
canopy_2015['has_inverter_fault'] = ~canopy_2015['label_inverter'].isin(NON_FAULT_VALUES)
canopy_2015['has_any_fault']      = canopy_2015['has_string_fault'] | canopy_2015['has_inverter_fault']
canopy_2015['fault_cooccurrence'] = canopy_2015['has_string_fault'] & canopy_2015['has_inverter_fault']

# Summary
print("=== Fault Indicator Summary (2015) ===\n")
print(f"has_string_fault   : {canopy_2015['has_string_fault'].sum()}")
print(f"has_inverter_fault : {canopy_2015['has_inverter_fault'].sum()}")
print(f"has_any_fault      : {canopy_2015['has_any_fault'].sum()}")
print(f"fault_cooccurrence : {canopy_2015['fault_cooccurrence'].sum()}")

print(f"\nAs % of evaluable rows (Op_Mask=1): "
      f"{canopy_2015.loc[canopy_2015['Op_Mask']==1, 'has_any_fault'].mean() * 100:.2f}%")

=== Fault Indicator Summary (2015) ===

has_string_fault   : 11146
has_inverter_fault : 10380
has_any_fault      : 20801
fault_cooccurrence : 725

As % of evaluable rows (Op_Mask=1): 10.92%


In [8]:
#2016
NON_FAULT_VALUES = ['NORMAL', 'NOT_EVALUABLE', 'MISSING_DATA', 'INSUFFICIENT_DATA']

canopy_2016['has_string_fault']   = ~canopy_2016['label_string'].isin(NON_FAULT_VALUES)
canopy_2016['has_inverter_fault'] = ~canopy_2016['label_inverter'].isin(NON_FAULT_VALUES)
canopy_2016['has_any_fault']      = canopy_2016['has_string_fault'] | canopy_2016['has_inverter_fault']
canopy_2016['fault_cooccurrence'] = canopy_2016['has_string_fault'] & canopy_2016['has_inverter_fault']

# Summary
print("=== Fault Indicator Summary (2016) ===\n")
print(f"has_string_fault   : {canopy_2016['has_string_fault'].sum()}")
print(f"has_inverter_fault : {canopy_2016['has_inverter_fault'].sum()}")
print(f"has_any_fault      : {canopy_2016['has_any_fault'].sum()}")
print(f"fault_cooccurrence : {canopy_2016['fault_cooccurrence'].sum()}")

print(f"\nAs % of evaluable rows (Op_Mask=1): "
      f"{canopy_2016.loc[canopy_2016['Op_Mask']==1, 'has_any_fault'].mean() * 100:.2f}%")

=== Fault Indicator Summary (2016) ===

has_string_fault   : 7610
has_inverter_fault : 11428
has_any_fault      : 18442
fault_cooccurrence : 596

As % of evaluable rows (Op_Mask=1): 9.54%


In [9]:
#2017
NON_FAULT_VALUES = ['NORMAL', 'NOT_EVALUABLE', 'MISSING_DATA', 'INSUFFICIENT_DATA']

canopy_2017['has_string_fault']   = ~canopy_2017['label_string'].isin(NON_FAULT_VALUES)
canopy_2017['has_inverter_fault'] = ~canopy_2017['label_inverter'].isin(NON_FAULT_VALUES)
canopy_2017['has_any_fault']      = canopy_2017['has_string_fault'] | canopy_2017['has_inverter_fault']
canopy_2017['fault_cooccurrence'] = canopy_2017['has_string_fault'] & canopy_2017['has_inverter_fault']

# Summary
print("=== Fault Indicator Summary (2017) ===\n")
print(f"has_string_fault   : {canopy_2017['has_string_fault'].sum()}")
print(f"has_inverter_fault : {canopy_2017['has_inverter_fault'].sum()}")
print(f"has_any_fault      : {canopy_2017['has_any_fault'].sum()}")
print(f"fault_cooccurrence : {canopy_2017['fault_cooccurrence'].sum()}")

print(f"\nAs % of evaluable rows (Op_Mask=1): "
      f"{canopy_2017.loc[canopy_2017['Op_Mask']==1, 'has_any_fault'].mean() * 100:.2f}%")

=== Fault Indicator Summary (2017) ===

has_string_fault   : 6725
has_inverter_fault : 16768
has_any_fault      : 22951
fault_cooccurrence : 542

As % of evaluable rows (Op_Mask=1): 11.96%


In [10]:
#2018
NON_FAULT_VALUES = ['NORMAL', 'NOT_EVALUABLE', 'MISSING_DATA', 'INSUFFICIENT_DATA']

canopy_2018['has_string_fault']   = ~canopy_2018['label_string'].isin(NON_FAULT_VALUES)
canopy_2018['has_inverter_fault'] = ~canopy_2018['label_inverter'].isin(NON_FAULT_VALUES)
canopy_2018['has_any_fault']      = canopy_2018['has_string_fault'] | canopy_2018['has_inverter_fault']
canopy_2018['fault_cooccurrence'] = canopy_2018['has_string_fault'] & canopy_2018['has_inverter_fault']

# Summary
print("=== Fault Indicator Summary (2018) ===\n")
print(f"has_string_fault   : {canopy_2018['has_string_fault'].sum()}")
print(f"has_inverter_fault : {canopy_2018['has_inverter_fault'].sum()}")
print(f"has_any_fault      : {canopy_2018['has_any_fault'].sum()}")
print(f"fault_cooccurrence : {canopy_2018['fault_cooccurrence'].sum()}")

print(f"\nAs % of evaluable rows (Op_Mask=1): "
      f"{canopy_2018.loc[canopy_2018['Op_Mask']==1, 'has_any_fault'].mean() * 100:.2f}%")

=== Fault Indicator Summary (2018) ===

has_string_fault   : 5036
has_inverter_fault : 14942
has_any_fault      : 19534
fault_cooccurrence : 444

As % of evaluable rows (Op_Mask=1): 10.54%


#Step 2: Build Healthy Mask

## Objective

For training the **CS-LSTM**, we use **only confirmed normal operating data**.

The model should learn the expected power production of a healthy inverter, without being influenced by fault conditions.

During **training**, only normal samples are used.

During **inference**, the trained model is applied to the **entire dataset**, including both normal and faulty periods, to generate prediction residuals.

---

## Filtering Criteria

A timestep is considered **normal** only if all of the following conditions are satisfied:

1. **`Op_Mask == 1`**
   - The inverter is operating within normal irradiance conditions.

2. **`InvOpStatus_Avg > 0`**
   - The inverter is ON (not shut down or idle).

3. **`has_any_fault == 0`**
   - No inverter faults.
   - No string faults.

---

## Why?

The CS-LSTM is trained exclusively on healthy operating data so that it learns the system's **expected normal behavior**.

Later, when the model is evaluated on the full dataset, deviations between the predicted and measured AC power (prediction residuals) become indicators of potential anomalies and are used as input features for **Stage 2 (XGBoost fault classification)**.

In [11]:
def create_healthy_mask(df):

    return (
        (df["Op_Mask"] == 1) &
        (df["InvOpStatus_Avg"] > 0) &
        (~df["has_any_fault"])
    )


healthy_mask_2015 = create_healthy_mask(canopy_2015)
healthy_mask_2016 = create_healthy_mask(canopy_2016)

In [12]:
healthy_mask_2017 = create_healthy_mask(canopy_2017)
healthy_mask_2018 = create_healthy_mask(canopy_2018)

In [13]:
print(f"Healthy Operating Periods (2015): {healthy_mask_2015.sum()}")
print(f"Healthy Operating Periods (2016): {healthy_mask_2016.sum()}")

Healthy Operating Periods (2015): 169665
Healthy Operating Periods (2016): 174843


In [14]:
print(f"Healthy Operating Periods (2017): {healthy_mask_2017.sum()}")
print(f"Healthy Operating Periods (2018): {healthy_mask_2018.sum()}")

Healthy Operating Periods (2017): 168887
Healthy Operating Periods (2018): 165755


#Step 3: Primary and Contextual Features

## Excluded Features - Data Leakage

These variables are **never used** as primary features because they either encode the target directly or contain information that would not be available during inference in a real-world deployment.

```python
# Derived directly from InvPAC_kW_Avg — circular dependency
PR, PR_roll_15min_weighted, PR_roll_5min_weighted,
PR_roll_60min_weighted, PR_roll_std_15min,
power_deficit_kW

# Physics-based expected values — the model is supposed to learn this mapping.
# Providing them as inputs would make the regression task trivial.
P_expected_DC_kW, P_expected_AC_kW

# Labels and fault indicators — severe target leakage
label_power, label_string, label_inverter,
has_string_fault, has_inverter_fault, has_any_fault, fault_cooccurrence,
inv_any_fault_reported, inv_fault_count

# Fault flags — also leakage (all zero by definition during normal-operation training)
InvMainFault_Max, InvDriveFault_Max, InvVoltageFault_Max,
InvGridFault_Max, InvTempFault_Max, InvSystemFault_Max, InvSystemWarn_Max

# Operational status flags
InvOpStatus_Avg, Op_Mask, InvPVMStatus_Max,
inv_shutdown_power_flag, inv_shutdown_status_flag, PR_evaluable
```

---

## Excluded Features — Redundancy / Noise

```python
# Pyranometers — POA_fused is already a superior irradiance estimate
Pyra1_Wm2_Avg, Pyra2_Wm2_Avg, Pyra3_Wm2_Avg

# Reference cells — already used to construct POA_fused
RefCell1_Wm2_Avg, RefCell2_Wm2_Avg

# AC power meter measurements — redundant with inverter AC power
# They measure essentially the same quantity at a different point
PwrMtrIa_Avg, PwrMtrIb_Avg, PwrMtrIc_Avg,
PwrMtrFreq_Avg, PwrMtrVa_Avg, PwrMtrVb_Avg, PwrMtrVc_Avg,
PwrMtrP_kW_Avg, PwrMtrPF_Avg

# Three-phase AC measurements — redundant since InvPAC_kW_Avg is already the total AC power
InvVa_Avg, InvVb_Avg, InvVc_Avg,
InvIa_Avg, InvIb_Avg, InvIc_Avg

InvVDVoltage_V_Avg      # DC bus voltage; largely redundant with InvVDCin_Avg
InvVPVin_Avg            # PV input voltage; largely redundant with InvVDCin_Avg
InvFreq_Avg             # Grid frequency; does not determine expected PV power
InvEtot_kWh_Max         # Cumulative energy; not an instantaneous signal

# String-level deviation features — encode fault patterns rather than normal operation
I_string_1_dev .. I_string_7_dev,
I_string_1_rel_dev .. I_string_7_rel_dev,
I_string_1_expected .. I_string_7_expected,
R_string_1_dev .. R_string_7_dev,
Pnorm_string_1 .. Pnorm_string_7,
Inorm_string_1 .. Inorm_string_7,
I_rel_dev_max, I_rel_dev_std, I_rel_dev_max_roll5,
I_rel_dev_max_core, I_rel_dev_max_core_roll5,
I_string_cv_roll5

# Derived inverter-efficiency features
inv_efficiency, inv_efficiency_clipped, inv_efficiency_roll5,
DC_power_discrepancy, DC_power_discrepancy_pct, inv_loading_ratio
```

---

## Primary Features ($x_t$) — 8 Features

These variables closely follow the methodology proposed by **Baruah et al.** and capture the underlying physical behavior of the PV system.

```python
PRIMARY_FEATURES = [
    # Irradiance — primary driver of PV generation
    'POA_fused',            # Plane-of-Array irradiance (W/m²)

    # Temperature — affects module efficiency
    'AmbTemp_C_Avg',        # Ambient temperature (°C)
    'module_temp',          # Module backsheet temperature (°C)

    # DC-side electrical measurements
    'InvVDCin_Avg',         # DC input voltage (V)
    'InvIDCin_Avg',         # DC input current (A)
    'InvPDC_kW_Avg',        # DC power (kW)

    # Array-level summary
    'I_string_mean',        # Mean string current (A)

    # Autoregressive input
    'InvPAC_kW_Avg_lag1',   # Previous AC power (kW), created using shift(1)
]

TARGET = 'InvPAC_kW_Avg'
```

`I_string_mean` is retained because it summarizes the overall operating level of the PV array without explicitly encoding fault information. Unlike string deviation features, the mean current represents normal operating behavior rather than abnormal patterns.

---

## RTD Temperature Sensors — Excluded

```text
RTD_C_Avg_1, RTD_C_Avg_2, RTD_C_Avg_4,
RTD_C_Avg_8, RTD_C_Avg_9, RTD_C_Avg_10
```

These variables correspond to module/backsheet temperatures measured by individual RTD sensors. Since the dataset already includes the reliable `module_temp` measurement (SEWS Module Temperature), the RTD sensors provide little additional information while increasing noise and model complexity. Consequently, they are excluded from the final feature set.

---

## Contextual Features ($z_t$) — 4 Features

These cyclical encodings provide seasonal and diurnal context to the Contextual Sparse LSTM.

```python
CONTEXTUAL_FEATURES = [
    'sin_doy',      # sin(2π × day_of_year / 365)
    'cos_doy',      # cos(2π × day_of_year / 365)
    'sin_hour',     # sin(2π × time_of_day / 24)
    'cos_hour',     # cos(2π × time_of_day / 24)
]
```

Although the original architecture also discusses STL residuals as contextual variables, they are omitted in this implementation. The four cyclical encodings sufficiently capture seasonal variation and time-of-day effects, while STL-based features can be evaluated later as part of an ablation study.

`z_t = [sin_doy, cos_doy, sin_hour, cos_hour]`

The vector $z_t$ is used to build the polynomial basis $Gamma(z_t)$, which is combined with the operational features inside the **CS-LSTM**.

In [15]:
# Feature Selection & Contextual Encoding

PRIMARY_FEATURES = [
    'POA_fused',
    'AmbTemp_C_Avg',
    'module_temp',
    'InvVDCin_Avg',
    'InvIDCin_Avg',
    'InvPDC_kW_Avg',
    'InvPAC_kW_Avg_lag1',
]

CONTEXTUAL_FEATURES = [
    'sin_doy',
    'cos_doy',
    'sin_hour',
    'cos_hour',
]

TARGET = 'InvPAC_kW_Avg'


def add_features(df):
    df = df.copy()

    # Autoregressive feature
    df['InvPAC_kW_Avg_lag1'] = df['InvPAC_kW_Avg'].shift(1)

    # Contextual features
    if 'TIMESTAMP' in df.columns:
      df = df.set_index(pd.to_datetime(df['TIMESTAMP']))

    doy  = df.index.day_of_year.values
    hour = df.index.hour.values + df.index.minute.values / 60.0

    df['sin_doy']  = np.sin(2 * np.pi * doy  / 365.0)
    df['cos_doy']  = np.cos(2 * np.pi * doy  / 365.0)
    df['sin_hour'] = np.sin(2 * np.pi * hour / 24.0)
    df['cos_hour'] = np.cos(2 * np.pi * hour / 24.0)

    return df

In [16]:
canopy_2015 = add_features(canopy_2015)
canopy_2016 = add_features(canopy_2016)

In [17]:
canopy_2017 = add_features(canopy_2017)
canopy_2018 = add_features(canopy_2018)

In [18]:
canopy_2015['InvPAC_kW_Avg_lag1'] = canopy_2015['InvPAC_kW_Avg_lag1'].bfill()
canopy_2016['InvPAC_kW_Avg_lag1'] = canopy_2016['InvPAC_kW_Avg_lag1'].bfill()
canopy_2017['InvPAC_kW_Avg_lag1'] = canopy_2017['InvPAC_kW_Avg_lag1'].bfill()
canopy_2018['InvPAC_kW_Avg_lag1'] = canopy_2018['InvPAC_kW_Avg_lag1'].bfill()

#Step 4: Segment Analysis

The CS-LSTM autoencoder requires **contiguous, uninterrupted stretches** of healthy
operation to learn a reliable baseline of normal PV behavior. A single fault-free
timestamp is not enough — the model needs sequences of consecutive healthy readings
long enough to fill a full input window (`SEQ_LEN = 60` minutes).

This step identifies, extracts, and filters those usable stretches from the
`healthy_mask` computed in Step 2.

**4.1 Segment identification (`analyze_segments`)**
Using a cumulative-sum trick on mask transitions (`mask != mask.shift()`), the healthy
mask is partitioned into contiguous runs of `True`/`False` values. For each run of
healthy timestamps, we compute:
- Segment length distribution (min, max, mean)
- Number of continuous healthy segments
- How many segments are long enough to contain at least one full `SEQ_LEN`-sized window
  (`segment_length - seq_len + 1`, clipped at 0)

This is diagnostic only — it tells us how much usable data survives the windowing
requirement before we commit to extracting it.

In [19]:
def analyze_segments(mask, label, seq_len=60):

    groups = (mask != mask.shift()).cumsum()

    segment_lengths = (
        mask.groupby(groups)
            .sum()
    )

    segment_lengths = segment_lengths[segment_lengths > 0]

    usable = (segment_lengths - seq_len + 1).clip(lower=0)

    print(f"\n========== {label} ==========")
    print(f"Healthy timestamps            : {segment_lengths.sum():,}")
    print(f"Continuous healthy segments   : {len(segment_lengths):,}")
    print(f"Longest segment               : {segment_lengths.max():,}")
    print(f"Average segment               : {segment_lengths.mean():.1f}")
    print(f"Segments < {seq_len} samples  : {(usable==0).sum():,}")
    print(f"Usable windows (L={seq_len})  : {usable.sum():,}")

    return segment_lengths

In [20]:
segments_2015 = analyze_segments(healthy_mask_2015,"2015")
segments_2016 = analyze_segments(healthy_mask_2016,"2016")


========== 2015 ==========
Healthy timestamps            : 169,665
Continuous healthy segments   : 3,552
Longest segment               : 678
Average segment               : 47.8
Segments < 60 samples  : 2,956
Usable windows (L=60)  : 92,539

========== 2016 ==========
Healthy timestamps            : 174,843
Continuous healthy segments   : 4,107
Longest segment               : 662
Average segment               : 42.6
Segments < 60 samples  : 3,527
Usable windows (L=60)  : 92,366


In [21]:
segments_2017 = analyze_segments(healthy_mask_2017,"2017")
segments_2018 = analyze_segments(healthy_mask_2018,"2018")


========== 2017 ==========
Healthy timestamps            : 168,887
Continuous healthy segments   : 4,177
Longest segment               : 664
Average segment               : 40.4
Segments < 60 samples  : 3,579
Usable windows (L=60)  : 85,840

========== 2018 ==========
Healthy timestamps            : 165,755
Continuous healthy segments   : 3,891
Longest segment               : 691
Average segment               : 42.6
Segments < 60 samples  : 3,299
Usable windows (L=60)  : 85,470


**4.2 Segment extraction (`extract_segments`)**
The same grouping logic is used to physically slice the dataframe into a list of
separate healthy segments (each a contiguous `DataFrame`), rather than just counting
lengths.


In [22]:
#Extract Continuous Healthy Segments
def extract_segments(df, healthy_mask):
    groups = (healthy_mask != healthy_mask.shift()).cumsum()
    segments = []

    for _, segment in df.groupby(groups):
        if not healthy_mask.loc[segment.index[0]]:
            continue
        segments.append(segment.copy())
    return segments

In [23]:
segments2015 = extract_segments(
    canopy_2015,
    healthy_mask_2015
)

segments2016 = extract_segments(
    canopy_2016,
    healthy_mask_2016
)

In [24]:
print(f"2015: {len(segments2015)} segments")
print(f"2016: {len(segments2016)} segments")

2015: 3552 segments
2016: 4107 segments


In [25]:
segments2017 = extract_segments(
    canopy_2017,
    healthy_mask_2017
)

segments2018 = extract_segments(
    canopy_2018,
    healthy_mask_2018
)

In [26]:
print(f"2017: {len(segments2017)} segments")
print(f"2018: {len(segments2018)} segments")

2017: 4177 segments
2018: 3891 segments


**4.3 Boundary cleaning (`clean_segments`)**
The **first row of every segment is dropped**. This is necessary because
`InvPAC_kW_Avg_lag1` (added in Step 3) was computed via `shift(1)` on the full,
unfiltered dataframe — meaning the first row of any healthy segment carries a lag
value from the previous (possibly faulty or masked) timestamp. Removing that row
eliminates this boundary leakage.

In [27]:
#Remove first row of every segment
def clean_segments(segments):
    cleaned = []
    removed = 0

    for seg in segments:
        if len(seg) <= 1:
            continue
        cleaned.append(seg.iloc[1:].copy())
        removed += 1

    print(f"Removed first row from {removed:,} segments.")
    return cleaned

In [28]:
segments2015 = clean_segments(segments2015)
segments2016 = clean_segments(segments2016)

Removed first row from 3,117 segments.
Removed first row from 3,552 segments.


In [29]:
segments2017 = clean_segments(segments2017)
segments2018 = clean_segments(segments2018)

Removed first row from 3,687 segments.
Removed first row from 3,444 segments.


**4.4 Minimum-length filtering**
Finally, any segment shorter than `SEQ_LEN` (60 samples) is discarded, since it cannot
produce even a single valid training window.

**Result:** Healthy, contiguous, leakage-free segments — reduced from thousands of raw
mask segments (3,552 in 2015 / 4,107 in 2016) down to a smaller set of *usable* segments
(586 in 2015 / 570 in 2016) that are all safe to feed into the windowing step (Step 5).

In [30]:
SEQ_LEN = 60

segments2015 = [
    s for s in segments2015
    if len(s) >= SEQ_LEN
]

segments2016 = [
    s for s in segments2016
    if len(s) >= SEQ_LEN
]

In [31]:
print(f"2015: {len(segments2015)} segments")
print(f"2016: {len(segments2016)} segments")

2015: 586 segments
2016: 570 segments


In [32]:
SEQ_LEN = 60

segments2017 = [
    s for s in segments2017
    if len(s) >= SEQ_LEN
]

segments2018 = [
    s for s in segments2018
    if len(s) >= SEQ_LEN
]

In [33]:
print(f"2017: {len(segments2017)} segments")
print(f"2018: {len(segments2018)} segments")

2017: 589 segments
2018: 584 segments


#Results

In [34]:
print("=" * 60)
print("Healthy Segment Summary")
print("=" * 60)

print(f"2015:")
print(f"  Total healthy segments : {len(segments2015):,}")
print(f"  Total healthy samples  : {sum(len(s) for s in segments2015):,}")

print()

print(f"2016:")
print(f"  Total healthy segments : {len(segments2016):,}")
print(f"  Total healthy samples  : {sum(len(s) for s in segments2016):,}")

Healthy Segment Summary
2015:
  Total healthy segments : 586
  Total healthy samples  : 126,517

2016:
  Total healthy segments : 570
  Total healthy samples  : 125,416


In [35]:
import pandas as pd

lengths2015 = pd.Series([len(s) for s in segments2015])
lengths2016 = pd.Series([len(s) for s in segments2016])

print("========== 2015 ==========")
print(lengths2015.describe())

print("\n========== 2016 ==========")
print(lengths2016.describe())

========== 2015 ==========
count    586.000000
mean     215.899317
std      150.829723
min       60.000000
25%       84.250000
50%      164.500000
75%      322.000000
max      677.000000
dtype: float64

========== 2016 ==========
count    570.00000
mean     220.02807
std      148.13755
min       60.00000
25%       88.00000
50%      169.50000
75%      331.00000
max      661.00000
dtype: float64


In [36]:
lengths2015 = pd.Series([len(s) for s in segments2015])

print("Top 10 longest healthy segments (2015):")

display(
    lengths2015.sort_values(ascending=False)
               .head(10)
               .rename("Length")
               .to_frame()
)

Top 10 longest healthy segments (2015):


,Length
199,677
236,670
197,660
177,639
150,628
237,606
129,584
395,579
300,578
411,575


In [37]:
segment_id = 300

segment = segments2015[segment_id]

print(f"Segment {segment_id}")
print(f"Rows      : {len(segment)}")
print(f"Start     : {segment.index[0]}")
print(f"End       : {segment.index[-1]}")

display(segment)

Segment 300
Rows      : 578
Start     : 2015-07-07 07:10:00-05:00
End       : 2015-07-07 16:47:00-05:00


,Pyra1_Wm2_Avg,Pyra2_Wm2_Avg,Pyra3_Wm2_Avg,RefCell1_Wm2_Avg,RefCell2_Wm2_Avg,AmbTemp_C_Avg,InvVDVoltage_V_Avg,InvVa_Avg,InvVb_Avg,InvVc_Avg,...,inv_shutdown_status_flag,has_string_fault,has_inverter_fault,has_any_fault,fault_cooccurrence,InvPAC_kW_Avg_lag1,sin_doy,cos_doy,sin_hour,cos_hour
TIMESTAMP,,,,,,,,,,,,,,,,,,,,,
2015-07-07 07:10:00-05:00,105.034820,99.726780,104.92843,108.0,106.1,22.34,336.0,273.5,274.3,275.6,...,0,False,False,False,False,23.19,-0.094537,-0.995521,0.953717,-0.300706
2015-07-07 07:11:00-05:00,109.999886,104.898520,110.08793,112.8,111.2,22.34,336.3,273.5,274.1,275.3,...,0,False,False,False,False,24.22,-0.094537,-0.995521,0.952396,-0.304864
2015-07-07 07:12:00-05:00,114.964950,109.972690,115.35960,117.7,116.1,22.29,337.3,273.3,274.1,275.3,...,0,False,False,False,False,25.39,-0.094537,-0.995521,0.951057,-0.309017
2015-07-07 07:13:00-05:00,120.639330,115.534744,120.96774,122.0,121.3,22.33,337.5,273.3,274.0,275.2,...,0,False,False,False,False,26.56,-0.094537,-0.995521,0.949699,-0.313164
2015-07-07 07:14:00-05:00,127.259415,121.877450,127.47319,126.9,127.3,22.38,337.5,273.1,273.9,275.1,...,0,False,False,False,False,27.86,-0.094537,-0.995521,0.948324,-0.317305
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2015-07-07 16:43:00-05:00,460.391780,388.075740,365.14648,435.6,331.1,31.56,330.2,271.5,272.5,274.6,...,0,False,False,False,False,86.30,-0.094537,-0.995521,-0.944089,-0.329691
2015-07-07 16:44:00-05:00,454.008150,382.220950,359.65048,429.2,326.0,31.64,331.1,271.5,272.6,274.8,...,0,False,False,False,False,85.70,-0.094537,-0.995521,-0.945519,-0.325568
2015-07-07 16:45:00-05:00,448.924870,377.537080,355.27612,424.5,321.9,31.53,329.9,271.8,272.6,274.8,...,0,False,False,False,False,84.40,-0.094537,-0.995521,-0.946930,-0.321439


#Step 5: Windowing

With clean, contiguous healthy segments in hand (Step 4), this step converts them into
fixed-length sliding windows suitable for the CS-LSTM autoencoder — pairing each window
of primary + contextual features with a target value to predict.

##Window generation

For a single segment, a sliding window of length `SEQ_LEN = 60` (60 minutes of history)
is moved one step at a time across the segment. For each window position:
- `X_primary`: the `PRIMARY_FEATURES` values across the 60-step window
- `X_context`: the `CONTEXTUAL_FEATURES` values (cyclical time encodings) across the
  same window
- `y`: the target (`InvPAC_kW_Avg`) at `HORIZON = 1` step beyond the end of the window,
  i.e. the next-timestep value the model must predict

This produces a next-step forecasting setup: given the last 60 minutes of healthy
operation, predict the AC power at the following minute.

In [38]:
SEQ_LEN = 60      # 60 minutes
HORIZON = 1       # Predict next timestep

In [39]:
def create_windows_from_segment(
    segment,
    primary_features,
    contextual_features,
    target,
    seq_len=60,
    horizon=1
):
    X_primary = []
    X_context = []
    y = []
    n = len(segment)

    for start in range(n - seq_len - horizon + 1):
        end = start + seq_len

        X_primary.append(
            segment.iloc[start:end][primary_features].values)

        X_context.append(
            segment.iloc[start:end][contextual_features].values)

        y.append(
            segment.iloc[end + horizon - 1][target])

    return X_primary, X_context, y

##Training Set Construction

In [40]:
def create_training_dataset(
    segments,
    primary_features,
    contextual_features,
    target,
    seq_len=60,
    horizon=1
):

    Xp = []
    Xc = []
    y = []

    total_segments = len(segments)
    skipped = 0

    for segment in segments:

        if len(segment) < seq_len + horizon:
            skipped += 1
            continue

        xp, xc, yy = create_windows_from_segment(
            segment,
            primary_features,
            contextual_features,
            target,
            seq_len,
            horizon
        )

        Xp.extend(xp)
        Xc.extend(xc)
        y.extend(yy)

    Xp = np.asarray(Xp, dtype=np.float32)
    Xc = np.asarray(Xc, dtype=np.float32)
    y = np.asarray(y, dtype=np.float32)

    print("="*60)
    print("Sliding Window Summary")
    print("="*60)
    print(f"Healthy segments            : {total_segments:,}")
    print(f"Skipped (<{seq_len+1} rows) : {skipped:,}")
    print(f"Used segments               : {total_segments-skipped:,}")
    print(f"Generated windows           : {len(y):,}")
    print()

    print(f"Primary tensor   : {Xp.shape}")
    print(f"Context tensor   : {Xc.shape}")
    print(f"Target vector    : {y.shape}")

    return Xp, Xc, y

In [41]:
#2015: training
Xp2015, Xc2015, y2015 = create_training_dataset(
    segments2015,
    PRIMARY_FEATURES,
    CONTEXTUAL_FEATURES,
    TARGET,
    seq_len=SEQ_LEN,
    horizon=HORIZON
)

Sliding Window Summary
Healthy segments            : 586
Skipped (<61 rows) : 9
Used segments               : 577
Generated windows           : 91,357

Primary tensor   : (91357, 60, 7)
Context tensor   : (91357, 60, 4)
Target vector    : (91357,)


In [42]:
#2016: training
Xp2016, Xc2016, y2016 = create_training_dataset(
    segments2016,
    PRIMARY_FEATURES,
    CONTEXTUAL_FEATURES,
    TARGET,
    seq_len=SEQ_LEN,
    horizon=HORIZON
)

Sliding Window Summary
Healthy segments            : 570
Skipped (<61 rows) : 12
Used segments               : 558
Generated windows           : 91,216

Primary tensor   : (91216, 60, 7)
Context tensor   : (91216, 60, 4)
Target vector    : (91216,)


In [43]:
#2017: validation
Xp2017, Xc2017, y2017 = create_training_dataset(
    segments2017,
    PRIMARY_FEATURES,
    CONTEXTUAL_FEATURES,
    TARGET,
    seq_len=SEQ_LEN,
    horizon=HORIZON
)

Sliding Window Summary
Healthy segments            : 589
Skipped (<61 rows) : 10
Used segments               : 579
Generated windows           : 84,653

Primary tensor   : (84653, 60, 7)
Context tensor   : (84653, 60, 4)
Target vector    : (84653,)


In [44]:
#2018: testing
Xp2018, Xc2018, y2018 = create_training_dataset(
    segments2018,
    PRIMARY_FEATURES,
    CONTEXTUAL_FEATURES,
    TARGET,
    seq_len=SEQ_LEN,
    horizon=HORIZON
)

Sliding Window Summary
Healthy segments            : 584
Skipped (<61 rows) : 15
Used segments               : 569
Generated windows           : 84,294

Primary tensor   : (84294, 60, 7)
Context tensor   : (84294, 60, 4)
Target vector    : (84294,)


In [45]:
#Final Training Set
Xp_train = np.concatenate([Xp2015, Xp2016], axis=0)
Xc_train = np.concatenate([Xc2015, Xc2016], axis=0)
y_train = np.concatenate([y2015, y2016], axis=0)

print("="*60)
print("Final Training Dataset")
print("="*60)

print(f"Primary tensor : {Xp_train.shape}")
print(f"Context tensor : {Xc_train.shape}")
print(f"Targets        : {y_train.shape}")

Final Training Dataset
Primary tensor : (182573, 60, 7)
Context tensor : (182573, 60, 4)
Targets        : (182573,)


#Step 6: Scaling

In [46]:
from sklearn.preprocessing import StandardScaler

# Fit feature scaler on the 2015–2016 training data.
primary_scaler = StandardScaler()

Xp_train_2d = Xp_train.reshape(-1, len(PRIMARY_FEATURES))

BATCH_ROWS = 500_000

for start in range(0, len(Xp_train_2d), BATCH_ROWS):
    primary_scaler.partial_fit(
        Xp_train_2d[start:start + BATCH_ROWS]
    )

# Transform primary features in place to limit memory usage.
def scale_primary_inplace(X, scaler, batch_rows=500_000):
    X_2d = X.reshape(-1, X.shape[-1])

    for start in range(0, len(X_2d), batch_rows):
        scaler.transform(
            X_2d[start:start + batch_rows],
            copy=False
        )

scale_primary_inplace(Xp_train, primary_scaler)
scale_primary_inplace(Xp2017, primary_scaler)
scale_primary_inplace(Xp2018, primary_scaler)

# Fit target scaler on the training years.
target_scaler = StandardScaler()
target_scaler.fit(y_train.reshape(-1, 1))

y_train = target_scaler.transform(
    y_train.reshape(-1, 1)
).ravel().astype(np.float32)

y2017 = target_scaler.transform(
    y2017.reshape(-1, 1)
).ravel().astype(np.float32)

y2018 = target_scaler.transform(
    y2018.reshape(-1, 1)
).ravel().astype(np.float32)

print("Scaling completed.")
print("Training primary:", Xp_train.shape)
print("Validation primary:", Xp2017.shape)
print("Test primary:", Xp2018.shape)
print("Training target mean:", round(float(y_train.mean()), 4))
print("Training target std:", round(float(y_train.std()), 4))
print("Context features remain unscaled.")

Scaling completed.
Training primary: (182573, 60, 7)
Validation primary: (84653, 60, 7)
Test primary: (84294, 60, 7)
Training target mean: 0.0
Training target std: 1.0
Context features remain unscaled.


In [47]:
for name, X, y in [
    ("Train", Xp_train, y_train),
    ("Validation", Xp2017, y2017),
    ("Test", Xp2018, y2018),
]:
    print(f"\n{name}")
    print("Primary mean:", round(float(X.mean()), 4))
    print("Primary std:", round(float(X.std()), 4))
    print("Target mean:", round(float(y.mean()), 4))
    print("Target std:", round(float(y.std()), 4))
    print("Finite primary:", bool(np.isfinite(X).all()))
    print("Finite target:", bool(np.isfinite(y).all()))


Train
Primary mean: -0.0
Primary std: 1.0
Target mean: 0.0
Target std: 1.0
Finite primary: True
Finite target: True

Validation
Primary mean: -0.0326
Primary std: 0.9995
Target mean: -0.1003
Target std: 0.9912
Finite primary: True
Finite target: True

Test
Primary mean: -0.066
Primary std: 1.0793
Target mean: -0.1153
Target std: 1.0149
Finite primary: True
Finite target: True


# CS-LSTM Training

We define the contextual polynomial basis
We have four contextual features:
$$
z_t = [\sin(doy),\cos(doy),\sin(hour),\cos(hour)]
$$

We'll use a complete degree-2 polynomial expansion:
$$
\Theta(z_t)=[1,z_1,\ldots,z_4,z_1^2,z_1z_2,\ldots,z_4^2]
$$

This produces 15 basis terms: one constant, four linear terms, and ten quadratic terms.

In [ ]:
import torch
import torch.nn as nn
import numpy as np

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", DEVICE)

class PolynomialContextBasis(nn.Module):
    def forward(self, z):
        # z: (batch, 4)
        terms = [torch.ones_like(z[:, :1]), z]

        quadratic_terms = [
            z[:, i:i+1] * z[:, j:j+1]
            for i in range(z.shape[1])
            for j in range(i, z.shape[1])
        ]

        return torch.cat(terms + quadratic_terms, dim=-1)

Device: cuda


CS-LSTM cell
The central difference from an ordinary LSTM is that the primary input weights depend on context.
For each gate, the paper describes a transformation of the form:
$$
W_x(z_t)x_t = B\big(x_t\otimes\Theta(z_t)\big)
$$

Instead of using fixed input-to-gate weights, the model learns how those weights change with the contextual variables.
We'll implement all four gates together for computational efficiency.

In [ ]:
class CSLSTMCell(nn.Module):
    def __init__(self, input_size, context_size, hidden_size):
        super().__init__()

        self.input_size = input_size
        self.context_size = context_size
        self.hidden_size = hidden_size

        self.basis = PolynomialContextBasis()

        basis_size = 1 + context_size + context_size * (context_size + 1) // 2
        expanded_size = input_size * basis_size

        # Context-dependent primary-input contribution
        self.input_projection = nn.Linear(
            expanded_size, 4 * hidden_size, bias=False
        )

        # Standard recurrent hidden-state contribution
        self.hidden_projection = nn.Linear(
            hidden_size, 4 * hidden_size, bias=True
        )

    def forward(self, x, z, h, c):
        # x: (batch, input_size)
        # z: (batch, context_size)

        theta = self.basis(z)

        # Kronecker-style expansion:
        # (batch, input_size, basis_size)
        expanded = torch.einsum("bi,bj->bij", x, theta)
        expanded = expanded.reshape(x.size(0), -1)

        gates = (
            self.input_projection(expanded)
            + self.hidden_projection(h)
        )

        # Gate order: forget, input, candidate, output
        f, i, g, o = gates.chunk(4, dim=-1)

        f = torch.sigmoid(f)
        i = torch.sigmoid(i)
        g = torch.tanh(g)
        o = torch.sigmoid(o)

        c_new = f * c + i * g
        h_new = o * torch.tanh(c_new)

        return h_new, c_new

**CS-LSTM regressor**

In [ ]:
class CSLSTMRegressor(nn.Module):
    def __init__(
        self,
        input_size=8,
        context_size=4,
        hidden_size=20
    ):
        super().__init__()

        self.hidden_size = hidden_size

        self.cell = CSLSTMCell(
            input_size=input_size,
            context_size=context_size,
            hidden_size=hidden_size
        )

        self.output_layer = nn.Linear(hidden_size, 1)

    def forward(self, x_primary, x_context):
        batch_size, seq_len, _ = x_primary.shape

        h = x_primary.new_zeros(batch_size, self.hidden_size)
        c = x_primary.new_zeros(batch_size, self.hidden_size)

        for t in range(seq_len):
            h, c = self.cell(
                x_primary[:, t, :],
                x_context[:, t, :],
                h,
                c
            )

        return self.output_layer(h).squeeze(-1)

In [ ]:
# Derive dimensions directly from the actual data
INPUT_SIZE = Xp_train.shape[-1]
CONTEXT_SIZE = Xc_train.shape[-1]
SEQ_LENGTH = Xp_train.shape[1]
HIDDEN_SIZE = 20

# Verify consistency
assert INPUT_SIZE == len(PRIMARY_FEATURES)
assert CONTEXT_SIZE == len(CONTEXTUAL_FEATURES)
assert SEQ_LENGTH == SEQ_LEN

print(f"Primary features: {INPUT_SIZE}")
print(f"Contextual features: {CONTEXT_SIZE}")
print(f"Sequence length: {SEQ_LENGTH}")

# Initialize the model
model = CSLSTMRegressor(
    input_size=INPUT_SIZE,
    context_size=CONTEXT_SIZE,
    hidden_size=HIDDEN_SIZE
).to(DEVICE)

# Create dummy inputs with the CORRECT dimensions
dummy_primary = torch.randn(
    16, SEQ_LENGTH, INPUT_SIZE,
    device=DEVICE
)

dummy_context = torch.randn(
    16, SEQ_LENGTH, CONTEXT_SIZE,
    device=DEVICE
)

# Test forward pass
model.eval()

with torch.no_grad():
    dummy_output = model(dummy_primary, dummy_context)

assert dummy_output.shape == (16,)

print("\nModel verification successful!")
print("Output shape:", dummy_output.shape)
print(
    "Trainable parameters:",
    sum(p.numel() for p in model.parameters())
)

Primary features: 7
Contextual features: 4
Sequence length: 60

Model verification successful!
Output shape: torch.Size([16])
Trainable parameters: 10101


In [ ]:
for name, Xp, Xc, y in [
    ("Train", Xp_train, Xc_train, y_train),
    ("Validation", Xp2017, Xc2017, y2017),
    ("Test", Xp2018, Xc2018, y2018),
]:
    assert Xp.ndim == 3
    assert Xc.ndim == 3
    assert Xp.shape[0] == Xc.shape[0] == len(y)
    assert Xp.shape[1] == Xc.shape[1] == SEQ_LENGTH
    assert Xp.shape[2] == INPUT_SIZE
    assert Xc.shape[2] == CONTEXT_SIZE
    assert np.isfinite(Xp).all()
    assert np.isfinite(Xc).all()
    assert np.isfinite(y).all()

    print(f"{name}: OK — {len(y):,} windows")

Train: OK — 182,573 windows
Validation: OK — 84,653 windows
Test: OK — 84,294 windows


**DataLoaders**

In [ ]:
from torch.utils.data import TensorDataset, DataLoader

BATCH_SIZE = 128

train_dataset = TensorDataset(
    torch.from_numpy(Xp_train),
    torch.from_numpy(Xc_train),
    torch.from_numpy(y_train)
)

val_dataset = TensorDataset(
    torch.from_numpy(Xp2017),
    torch.from_numpy(Xc2017),
    torch.from_numpy(y2017)
)

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=0,
    pin_memory=(DEVICE.type == "cuda")
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=0,
    pin_memory=(DEVICE.type == "cuda")
)

print("Training batches:", len(train_loader))
print("Validation batches:", len(val_loader))

Training batches: 1427
Validation batches: 662


**Training Parameters**

In [ ]:
import copy
import time

LEARNING_RATE = 1e-3
MAX_EPOCHS = 30
PATIENCE = 5

criterion = nn.MSELoss()

optimizer = torch.optim.RMSprop(
    model.parameters(),
    lr=LEARNING_RATE
)

best_val_loss = float("inf")
best_model_state = None
epochs_without_improvement = 0

history = {
    "train_loss": [],
    "val_loss": []
}

**Training loop**

In [ ]:
for epoch in range(MAX_EPOCHS):
    start_time = time.time()

    # Training
    model.train()
    train_loss_sum = 0.0

    for xp_batch, xc_batch, y_batch in train_loader:
        xp_batch = xp_batch.to(DEVICE, non_blocking=True)
        xc_batch = xc_batch.to(DEVICE, non_blocking=True)
        y_batch = y_batch.to(DEVICE, non_blocking=True)

        optimizer.zero_grad(set_to_none=True)

        predictions = model(xp_batch, xc_batch)
        loss = criterion(predictions, y_batch)

        loss.backward()

        torch.nn.utils.clip_grad_norm_(
            model.parameters(),
            max_norm=1.0
        )

        optimizer.step()

        train_loss_sum += loss.item() * len(y_batch)

    train_loss = train_loss_sum / len(train_dataset)

    # Validation
    model.eval()
    val_loss_sum = 0.0

    with torch.no_grad():
        for xp_batch, xc_batch, y_batch in val_loader:
            xp_batch = xp_batch.to(DEVICE, non_blocking=True)
            xc_batch = xc_batch.to(DEVICE, non_blocking=True)
            y_batch = y_batch.to(DEVICE, non_blocking=True)

            predictions = model(xp_batch, xc_batch)
            loss = criterion(predictions, y_batch)

            val_loss_sum += loss.item() * len(y_batch)

    val_loss = val_loss_sum / len(val_dataset)

    history["train_loss"].append(train_loss)
    history["val_loss"].append(val_loss)

    elapsed = time.time() - start_time

    print(
        f"Epoch {epoch+1:02d}/{MAX_EPOCHS} | "
        f"Train MSE: {train_loss:.6f} | "
        f"Val MSE: {val_loss:.6f} | "
        f"Time: {elapsed:.1f}s",
        flush=True
    )

    # Early stopping
    if val_loss < best_val_loss - 1e-6:
        best_val_loss = val_loss
        best_model_state = copy.deepcopy(model.state_dict())
        epochs_without_improvement = 0

        torch.save(
            best_model_state,
            "/content/cslstm_best.pt"
        )
    else:
        epochs_without_improvement += 1

    if epochs_without_improvement >= PATIENCE:
        print(f"Early stopping at epoch {epoch+1}")
        break

# Restore best validation checkpoint
model.load_state_dict(best_model_state)

print("\nTraining completed.")
print(f"Best validation MSE: {best_val_loss:.6f}")

Epoch 01/30 | Train MSE: 0.724224 | Val MSE: 0.474667 | Time: 128.9s
Epoch 02/30 | Train MSE: 0.302112 | Val MSE: 0.196068 | Time: 123.3s
Epoch 03/30 | Train MSE: 0.144358 | Val MSE: 0.131664 | Time: 121.0s
Epoch 04/30 | Train MSE: 0.105973 | Val MSE: 0.111344 | Time: 122.8s
Epoch 05/30 | Train MSE: 0.090922 | Val MSE: 0.101729 | Time: 122.3s
Epoch 06/30 | Train MSE: 0.083231 | Val MSE: 0.096139 | Time: 122.1s
Epoch 07/30 | Train MSE: 0.078510 | Val MSE: 0.092207 | Time: 120.4s
Epoch 08/30 | Train MSE: 0.075191 | Val MSE: 0.089404 | Time: 121.8s
Epoch 09/30 | Train MSE: 0.072752 | Val MSE: 0.087289 | Time: 119.9s
Epoch 10/30 | Train MSE: 0.070898 | Val MSE: 0.085697 | Time: 121.5s
Epoch 11/30 | Train MSE: 0.069452 | Val MSE: 0.084327 | Time: 121.5s
Epoch 12/30 | Train MSE: 0.068349 | Val MSE: 0.083431 | Time: 121.2s
Epoch 13/30 | Train MSE: 0.067496 | Val MSE: 0.082835 | Time: 121.4s
Epoch 14/30 | Train MSE: 0.066829 | Val MSE: 0.082082 | Time: 119.9s
Epoch 15/30 | Train MSE: 0.066306 

#XGBoost Preparation

In [48]:
for year, df in [
    (2015, canopy_2015),
    (2016, canopy_2016),
    (2017, canopy_2017),
    (2018, canopy_2018),
]:
    print(f"\n{'='*60}")
    print(f"YEAR {year}")
    print(f"{'='*60}")

    label_cols = [
        c for c in df.columns
        if "label" in c.lower() or "fault" in c.lower()
    ]

    print("Label-related columns:", label_cols)

    for col in label_cols:
        if col == "has_any_fault":
            print(
                "\nhas_any_fault:",
                df[col].value_counts(dropna=False).to_dict()
            )
        elif df[col].dtype == "object" or str(df[col].dtype) == "category":
            print(f"\n{col}:")
            print(df[col].value_counts(dropna=False).head(20))


YEAR 2015
Label-related columns: ['InvMainFault_Max', 'InvDriveFault_Max', 'InvVoltageFault_Max', 'InvGridFault_Max', 'InvTempFault_Max', 'InvSystemFault_Max', 'label_power', 'label_string', 'inv_any_fault_reported', 'inv_fault_count', 'label_inverter', 'has_string_fault', 'has_inverter_fault', 'has_any_fault', 'fault_cooccurrence']

label_power:
label_power
NOT_EVALUABLE                382291
NORMAL                       129478
CRITICAL_UNDERPERFORMANCE      5448
INSUFFICIENT_DATA              5434
UNDERPERFORMANCE               2168
SEVERE_UNDERPERFORMANCE         658
SUSPICIOUSLY_HIGH               123
Name: count, dtype: int64

label_string:
label_string
NOT_EVALUABLE          382291
NORMAL                 132144
STRING_IMBALANCE        10870
OPEN_CIRCUIT_STRING       276
MISSING_DATA               19
Name: count, dtype: int64

label_inverter:
label_inverter
NOT_EVALUABLE                335122
NORMAL                       180098
INVERTER_EFFICIENCY_FAULT      9563
INVERTER_SHUTDOW

In [49]:
LABEL_COLUMNS = [
    "label_power",
    "label_string",
    "label_inverter"
]

UNKNOWN_LABELS = {
    "NOT_EVALUABLE",
    "INSUFFICIENT_DATA",
    "MISSING_DATA"
}

def create_binary_fault_target(df):
    df = df.copy()

    labels = df[LABEL_COLUMNS].astype("string").apply(
        lambda col: col.str.strip().str.upper()
    )

    is_normal = labels.eq("NORMAL").fillna(False)

    is_unknown = (
        labels.isna()
        | labels.isin(UNKNOWN_LABELS)
    ).fillna(True)

    # Everything explicitly labeled as a fault
    is_fault = ~(is_normal | is_unknown)

    any_fault = is_fault.any(axis=1)
    all_normal = is_normal.all(axis=1)

    df["binary_fault_target"] = np.select(
        [any_fault, all_normal],
        [1.0, 0.0],
        default=np.nan
    )

    return df

In [50]:
canopy_2015 = create_binary_fault_target(canopy_2015)
canopy_2016 = create_binary_fault_target(canopy_2016)
canopy_2017 = create_binary_fault_target(canopy_2017)
canopy_2018 = create_binary_fault_target(canopy_2018)

for year, df in [
    (2015, canopy_2015),
    (2016, canopy_2016),
    (2017, canopy_2017),
    (2018, canopy_2018),
]:
    print(f"\nYEAR {year}")

    print(
        df["binary_fault_target"]
        .map({0.0: "NORMAL", 1.0: "FAULT"})
        .fillna("UNKNOWN")
        .value_counts()
    )


YEAR 2015
binary_fault_target
UNKNOWN    385859
NORMAL     118607
FAULT       21134
Name: count, dtype: int64

YEAR 2016
binary_fault_target
UNKNOWN    383142
NORMAL     124787
FAULT       19111
Name: count, dtype: int64

YEAR 2017
binary_fault_target
UNKNOWN    384411
NORMAL     116954
FAULT       24235
Name: count, dtype: int64

YEAR 2018
binary_fault_target
UNKNOWN    392521
NORMAL     112538
FAULT       20048
Name: count, dtype: int64


In [51]:
IRR_THRESHOLD = 100.0

def create_classification_mask(df):
    return (
        (df["POA_fused"] >= IRR_THRESHOLD)
        & (df["Op_Mask"] == 1)
        & df["binary_fault_target"].notna()
    )

for year, df in [
    (2015, canopy_2015),
    (2016, canopy_2016),
    (2017, canopy_2017),
    (2018, canopy_2018),
]:
    mask = create_classification_mask(df)

    counts = (
        df.loc[mask, "binary_fault_target"]
        .value_counts()
    )

    normal = int(counts.get(0.0, 0))
    fault = int(counts.get(1.0, 0))

    print(
        f"{year} | "
        f"Normal: {normal:,} | "
        f"Fault: {fault:,} | "
        f"Fault ratio: {fault / max(normal + fault, 1):.2%}"
    )

2015 | Normal: 118,607 | Fault: 21,134 | Fault ratio: 15.12%
2016 | Normal: 124,787 | Fault: 19,111 | Fault ratio: 13.28%
2017 | Normal: 116,954 | Fault: 24,235 | Fault ratio: 17.16%
2018 | Normal: 112,538 | Fault: 20,048 | Fault ratio: 15.12%


In [52]:
SEQ_LEN = 60

def create_classification_windows(
    df,
    primary_features,
    contextual_features,
    target,
    seq_len=60
):
    df = df.sort_index().copy()

    required_features = (
        primary_features
        + contextual_features
        + [target]
    )

    # Valid numerical measurements
    valid_features = (
        df[required_features]
        .notna()
        .all(axis=1)
    )

    # Input rows must satisfy operating conditions
    valid_operating = (
        (df["POA_fused"] >= 100.0)
        & (df["Op_Mask"] == 1)
    )

    # Target must have a known classification label
    valid_target = df["binary_fault_target"].notna()

    Xp, Xc = [], []
    y_power, y_fault = [], []
    timestamps = []

    # Iterate over possible target positions
    for end in range(seq_len, len(df)):
        start = end - seq_len

        window = df.iloc[start:end]
        target_row = df.iloc[end]

        # Check timestamp continuity (1-minute resolution)
        expected_index = pd.date_range(
            start=window.index[0],
            periods=seq_len + 1,
            freq="min"
        )

        actual_index = df.index[start:end + 1]

        if not expected_index.equals(actual_index):
            continue

        # All input measurements must be valid
        if not valid_features.iloc[start:end].all():
            continue

        # All input rows must satisfy operating conditions
        if not valid_operating.iloc[start:end].all():
            continue

        # Target row must also be eligible
        if not valid_features.iloc[end]:
            continue

        if not valid_operating.iloc[end]:
            continue

        if not valid_target.iloc[end]:
            continue

        Xp.append(
            window[primary_features].to_numpy(dtype=np.float32)
        )

        Xc.append(
            window[contextual_features].to_numpy(dtype=np.float32)
        )

        y_power.append(float(target_row[target]))
        y_fault.append(int(target_row["binary_fault_target"]))
        timestamps.append(df.index[end])

    return (
        np.asarray(Xp, dtype=np.float32),
        np.asarray(Xc, dtype=np.float32),
        np.asarray(y_power, dtype=np.float32),
        np.asarray(y_fault, dtype=np.int64),
        pd.DatetimeIndex(timestamps)
    )

In [53]:
classification_data = {}

for year, df in [
    (2015, canopy_2015),
    (2016, canopy_2016),
    (2017, canopy_2017),
    (2018, canopy_2018),
]:
    print(f"\nProcessing {year}...")

    Xp, Xc, y_power, y_fault, timestamps = (
        create_classification_windows(
            df,
            PRIMARY_FEATURES,
            CONTEXTUAL_FEATURES,
            TARGET,
            seq_len=SEQ_LEN
        )
    )

    classification_data[year] = {
        "Xp": Xp,
        "Xc": Xc,
        "y_power": y_power,
        "y_fault": y_fault,
        "timestamps": timestamps
    }

    print("Primary shape:", Xp.shape)
    print("Context shape:", Xc.shape)
    print("Normal windows:", np.sum(y_fault == 0))
    print("Fault windows:", np.sum(y_fault == 1))


Processing 2015...
Primary shape: (129448, 60, 7)
Context shape: (129448, 60, 4)
Normal windows: 110713
Fault windows: 18735

Processing 2016...
Primary shape: (132658, 60, 7)
Context shape: (132658, 60, 4)
Normal windows: 116701
Fault windows: 15957

Processing 2017...
Primary shape: (129468, 60, 7)
Context shape: (129468, 60, 4)
Normal windows: 109107
Fault windows: 20361

Processing 2018...
Primary shape: (121564, 60, 7)
Context shape: (121564, 60, 4)
Normal windows: 104655
Fault windows: 16909


In [ ]:
import numpy as np
import torch

model.eval()

@torch.no_grad()
def get_residuals(data, batch_size=256):

    Xp = data["Xp"]
    Xc = data["Xc"]

    predictions = []

    for i in range(0, len(Xp), batch_size):

        xp_batch = Xp[i:i+batch_size]
        xc_batch = Xc[i:i+batch_size]

        # Scale primary features
        xp_scaled = primary_scaler.transform(
            xp_batch.reshape(-1, xp_batch.shape[-1])
        ).reshape(xp_batch.shape)

        xp_tensor = torch.tensor(
            xp_scaled,
            dtype=torch.float32,
            device=DEVICE
        )

        xc_tensor = torch.tensor(
            xc_batch,
            dtype=torch.float32,
            device=DEVICE
        )

        pred = model(xp_tensor, xc_tensor)

        predictions.append(pred.cpu().numpy())

    predictions_scaled = np.concatenate(predictions)

    predictions_kw = target_scaler.inverse_transform(
        predictions_scaled.reshape(-1, 1)
    ).ravel()

    residuals = data["y_power"] - predictions_kw

    return residuals


for year, data in classification_data.items():
    print(f"Processing {year}...")

    data["residuals"] = get_residuals(data)

    print("Residuals:", data["residuals"].shape)

In [ ]:
from xgboost import XGBClassifier

# Training data
X_train = np.concatenate([
    classification_data[2015]["residuals"],
    classification_data[2016]["residuals"]
]).reshape(-1, 1)

y_train = np.concatenate([
    classification_data[2015]["y_fault"],
    classification_data[2016]["y_fault"]
])

# Validation data
X_val = classification_data[2017]["residuals"].reshape(-1, 1)
y_val = classification_data[2017]["y_fault"]

# Test data
X_test = classification_data[2018]["residuals"].reshape(-1, 1)
y_test = classification_data[2018]["y_fault"]

# XGBoost
xgb_model = XGBClassifier(
    n_estimators=100,
    max_depth=4,
    learning_rate=0.01,
    objective="binary:logistic",
    eval_metric="logloss",
    random_state=42
)

xgb_model.fit(X_train, y_train)

print("XGBoost training completed!")

In [ ]:
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix
)

y_pred = xgb_model.predict(X_test)

print("=== XGBoost Test Results (2018) ===")

print("Accuracy :", accuracy_score(y_test, y_pred))
print("Precision:", precision_score(y_test, y_pred, zero_division=0))
print("Recall   :", recall_score(y_test, y_pred, zero_division=0))
print("F1-score :", f1_score(y_test, y_pred, zero_division=0))

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred))